In [ ]:
# Standalone cell: Similarity of all PAMAP2 activities vs Walking(4) & Rope Jumping(24)

import os, re, math, glob
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import pairwise_kernels

# --- Torch only for device check (optional); the rest is pure numpy/pandas ---
try:
    import torch
except Exception:
    torch = None

# ---- If you have your Dataset class file available, you can use it to resolve the path. ----
# from dataset import Dataset
# ds = Dataset(dataset_name="pamap2", seed=42, device="cpu")
# DATA_DIR = Path(ds.data_path)

# Otherwise, set your PAMAP2 root manually (must contain Protocol/ and/or Optional/):
DATA_DIR = Path("/home/epigou/cs_9170_project/datasets/PAMAP2_Dataset")

# ---------------------------
# Config (matches your split_pamap2 defaults)
# ---------------------------
FS = 100
WIN_SECONDS  = 5.0
STEP_SECONDS = 2.5
WIN  = int(WIN_SECONDS * FS)
STEP = int(STEP_SECONDS * FS)

IMU_POS = ["hand", "chest", "ankle"]
ACC16   = ["acc16g_x","acc16g_y","acc16g_z"]
GYR     = ["gyro_x","gyro_y","gyro_z"]
DROP_MAGNETOMETERS = True
USE_VECTOR_NORMS   = True
STATS = ("std","rms")  # same as your default list

# PCA dimension for embedding used in distances
PCA_COMPONENTS = 6

# Cap samples per class for MMD (keeps it fast)
MAX_PER_CLASS_FOR_MMD = 4000

# RBF kernel widths to stabilize MMD
RBF_GAMMAS = [1e-3, 5e-3, 1e-2, 5e-2, 1e-1]

# ---------------------------
# Utility: file parsing, columns, loading
# ---------------------------
def colnames():
    cols = ["timestamp", "activity_id", "heart_rate"]
    sub = [
        "temp","acc16g_x","acc16g_y","acc16g_z","acc6g_x","acc6g_y","acc6g_z",
        "gyro_x","gyro_y","gyro_z","mag_x","mag_y","mag_z","orient_w","orient_x","orient_y","orient_z"
    ]
    for p in IMU_POS:
        cols += [f"{p}_{s}" for s in sub]
    return cols  # 54 total

def parse_sid(path: Path):
    m = re.search(r"subject(\d+)", path.stem.lower())
    return m.group(1) if m else "unknown"

def list_dat_files(data_dir: Path):
    files = []
    for sub in ["Protocol", "Optional", "protocol", "optional"]:
        d = data_dir / sub
        if d.exists():
            files += sorted(d.glob("subject*.dat"))
    return files

# ---------------------------
# Load raw PAMAP2
# ---------------------------
files = list_dat_files(DATA_DIR)
if not files:
    raise FileNotFoundError(f"No .dat files found under {DATA_DIR} (looked in Protocol/ and Optional/).")

dfs = []
for f in files:
    df = pd.read_csv(
        f, sep=r"\s+", header=None, names=colnames(),
        engine="python", na_values=["NaN","nan"]
    )
    df["subject_id"] = parse_sid(f)
    dfs.append(df)
raw = pd.concat(dfs, ignore_index=True)

# Keep numeric columns for interpolation (exclude labels)
num_cols = raw.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c not in ("activity_id",)]
raw = raw.sort_values(["subject_id", "timestamp"]).groupby("subject_id", group_keys=False).apply(
    lambda g: g.assign(**{c: g[c].interpolate(limit_direction="both") for c in num_cols})
)

# ---------------------------
# Base channels: heart + (acc16g, gyro [, mag]) as vector norms (like your code)
# ---------------------------
keep_triplets = [("acc", ACC16), ("gyr", GYR)]
if not DROP_MAGNETOMETERS:
    keep_triplets.append(("mag", ["mag_x","mag_y","mag_z"]))

base_cols = ["heart_rate"]
if USE_VECTOR_NORMS:
    for p in IMU_POS:
        for name, axes in keep_triplets:
            cols = [f"{p}_{a}" for a in axes]
            raw[f"{p}_{name}_norm"] = np.sqrt((raw[cols].values ** 2).sum(axis=1))
            base_cols.append(f"{p}_{name}_norm")
else:
    for p in IMU_POS:
        for _, axes in keep_triplets:
            for a in axes:
                base_cols.append(f"{p}_{a}")

# ---------------------------
# Windowing → features (std, rms)
# Label an entire window with the majority activity_id (mode); if tie, round the mean.
# ---------------------------
def window_features(df_windows: pd.DataFrame, base_cols: list[str]) -> dict:
    feats = {}
    if "std" in STATS:
        feats.update(df_windows[base_cols].std(ddof=1).add_suffix("__std").to_dict())
    if "rms" in STATS:
        feats.update((np.sqrt((df_windows[base_cols]**2).mean())).add_suffix("__rms").to_dict())
    return feats

rows, labels, subjects = [], [], []
raw = raw.sort_values(["subject_id", "timestamp"]).reset_index(drop=True)

for sid, g in raw.groupby("subject_id", sort=False):
    g = g.reset_index(drop=True)
    n = len(g)
    for start in range(0, max(0, n - WIN + 1), STEP):
        w = g.iloc[start:start + WIN]
        if len(w) < WIN:
            continue
        # Assign window label by the most frequent activity_id
        y_win = int(np.bincount(w["activity_id"].astype(int)).argmax())
        feats = window_features(w, base_cols)
        feats["subject_id"] = sid
        rows.append(feats); labels.append(y_win); subjects.append(sid)

feat_df = pd.DataFrame(rows).fillna(0.0)
y_all = np.asarray(labels, dtype=int)

# ---------------------------
# Standardize numeric features; PCA for embedding
# ---------------------------
meta_cols = ["subject_id"]
feature_cols = [c for c in feat_df.columns if c not in meta_cols]

scaler = StandardScaler()
Xz = scaler.fit_transform(feat_df[feature_cols].values)

pca = PCA(n_components=PCA_COMPONENTS, svd_solver="full", random_state=42)
Xp = pca.fit_transform(Xz)

# ---------------------------
# Similarity metrics
# ---------------------------
WALK_ID, ROPE_ID = 4, 24
unique_acts = np.unique(y_all)
# Exclude the two base classes when searching for a third
candidates = [a for a in unique_acts if a not in (WALK_ID, ROPE_ID)]

def class_mask(act_id):
    return (y_all == act_id)

def centroid(X):
    return X.mean(axis=0)

def cov_reg(X, eps=1e-3):
    C = np.cov(X.T)
    # ensure positive definite with ridge
    return C + eps * np.eye(C.shape[0])

def bhattacharyya_distance(Xa, Xb):
    # Gaussian approx BD: 1/8 (μa-μb)^T Σ^{-1} (μa-μb) + 1/2 ln( det(Σ) / sqrt(det(Σa)det(Σb)) )
    # where Σ = (Σa + Σb)/2
    ma, mb = centroid(Xa), centroid(Xb)
    Sa, Sb = cov_reg(Xa), cov_reg(Xb)
    S = 0.5 * (Sa + Sb)
    # Solve for Mahalanobis using linear solver for stability
    dmu = (ma - mb)
    try:
        term1 = 0.125 * dmu @ np.linalg.solve(S, dmu)
        detS  = max(1e-12, np.linalg.det(S))
        detSa = max(1e-12, np.linalg.det(Sa))
        detSb = max(1e-12, np.linalg.det(Sb))
        term2 = 0.5 * np.log(detS / math.sqrt(detSa * detSb))
        return float(term1 + term2)
    except np.linalg.LinAlgError:
        # Fallback to pseudo-inverse
        Sinv = np.linalg.pinv(S)
        term1 = 0.125 * dmu @ Sinv @ dmu
        detS  = max(1e-12, np.linalg.det(S + 1e-6*np.eye(S.shape[0])))
        detSa = max(1e-12, np.linalg.det(Sa + 1e-6*np.eye(Sa.shape[0])))
        detSb = max(1e-12, np.linalg.det(Sb + 1e-6*np.eye(Sb.shape[0])))
        term2 = 0.5 * np.log(detS / math.sqrt(detSa * detSb))
        return float(term1 + term2)

def mmd_rbf(X, Y, gammas):
    # Biased MMD^2 across multiple gammas, averaged (gives scale robustness)
    # MMD^2 = E[k(x,x')] + E[k(y,y')] - 2E[k(x,y)]
    # subsample if too large
    def _sample(A, max_n):
        if len(A) > max_n:
            idx = np.random.RandomState(42).choice(len(A), size=max_n, replace=False)
            return A[idx]
        return A
    Xs = _sample(X, MAX_PER_CLASS_FOR_MMD)
    Ys = _sample(Y, MAX_PER_CLASS_FOR_MMD)
    Kxx = pairwise_kernels(Xs, Xs, metric="rbf", gamma=None)
    Kyy = pairwise_kernels(Ys, Ys, metric="rbf", gamma=None)
    Kxy = pairwise_kernels(Xs, Ys, metric="rbf", gamma=None)

    # Recompute with multiple gammas more simply by re-calling pairwise_kernels
    vals = []
    for g in gammas:
        Kxx_g = pairwise_kernels(Xs, Xs, metric="rbf", gamma=g)
        Kyy_g = pairwise_kernels(Ys, Ys, metric="rbf", gamma=g)
        Kxy_g = pairwise_kernels(Xs, Ys, metric="rbf", gamma=g)
        m = len(Xs); n = len(Ys)
        val = (Kxx_g.sum() - np.trace(Kxx_g)) / (m*(m-1)+1e-12) \
            + (Kyy_g.sum() - np.trace(Kyy_g)) / (n*(n-1)+1e-12) \
            - 2.0 * Kxy_g.mean()
        vals.append(val)
    return float(np.mean(vals))

def euclid_centroid_dist(Xa, Xb):
    return float(np.linalg.norm(centroid(Xa) - centroid(Xb)))

# Pre-slice matrices
X_walk = Xp[class_mask(WALK_ID)]
X_rope = Xp[class_mask(ROPE_ID)]

# Build similarity table
rows = []
counts = {a: int(class_mask(a).sum()) for a in unique_acts}

for a in candidates:
    Xc = Xp[class_mask(a)]
    if len(Xc) < 5:
        # Skip tiny classes
        continue

    # Distances to walking and rope
    eu_w = euclid_centroid_dist(Xc, X_walk)
    eu_r = euclid_centroid_dist(Xc, X_rope)

    bd_w = bhattacharyya_distance(Xc, X_walk)
    bd_r = bhattacharyya_distance(Xc, X_rope)

    mmd_w = mmd_rbf(Xc, X_walk, RBF_GAMMAS)
    mmd_r = mmd_rbf(Xc, X_rope, RBF_GAMMAS)

    rows.append({
        "activity_id": a,
        "n_windows": counts[a],
        "euclid_to_walk": eu_w,
        "euclid_to_rope": eu_r,
        "bhattacharyya_to_walk": bd_w,
        "bhattacharyya_to_rope": bd_r,
        "mmd_to_walk": mmd_w,
        "mmd_to_rope": mmd_r,
        # A simple average-of-ranks DifficultyScore (lower distances => higher ranks => harder)
    })

df = pd.DataFrame(rows)

if len(df) == 0:
    print("No candidate activities found (besides 4 and 24). Check your data path or windowing params.")
else:
    # Rank each metric (ascending distance => higher difficulty => use ascending rank, then invert)
    # We'll compute mean of ranks across the 3 averaged metrics
    df["euclid_mean"] = (df["euclid_to_walk"] + df["euclid_to_rope"]) / 2.0
    df["bhatt_mean"]  = (df["bhattacharyya_to_walk"] + df["bhattacharyya_to_rope"]) / 2.0
    df["mmd_mean"]    = (df["mmd_to_walk"] + df["mmd_to_rope"]) / 2.0

    # Ranks: smaller is more similar, so invert to a “difficulty score” (higher is tougher)
    for col in ["euclid_mean", "bhatt_mean", "mmd_mean"]:
        df[f"rank_{col}"] = df[col].rank(method="average", ascending=True)

    # Convert ranks to 0..1 (1 = hardest), then average
    for col in ["rank_euclid_mean", "rank_bhatt_mean", "rank_mmd_mean"]:
        df[col + "_norm"] = (df[col] - df[col].min()) / max(1e-12, (df[col].max() - df[col].min()))

    df["DifficultyScore"] = df[["rank_euclid_mean_norm","rank_bhatt_mean_norm","rank_mmd_mean_norm"]].mean(axis=1)

    df_sorted = df.sort_values("DifficultyScore", ascending=False).reset_index(drop=True)

    # Nice compact view
    display_cols = [
        "activity_id","n_windows",
        "euclid_mean","bhatt_mean","mmd_mean",
        "DifficultyScore"
    ]
    print("Similarity of each activity to Walking(4) & Rope(24) — higher DifficultyScore => more confusable with both")
    display(df_sorted[display_cols].round(4))

    # Optional: also show top 5 suggestions
    print("\nTop 5 suggested third activities to make the task harder:")
    display(df_sorted[display_cols].head(5).round(4))


/tmp/ipykernel_45242/4207051848.py:96: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  raw = raw.sort_values(["subject_id", "timestamp"]).groupby("subject_id", group_keys=False).apply(


Similarity of each activity to Walking(4) & Rope(24) — higher DifficultyScore => more confusable with both


,activity_id,n_windows,euclid_mean,bhatt_mean,mmd_mean,DifficultyScore
0,1,993,8.6247,8.5943,0.7664,0.9778
1,2,1118,8.4851,7.7755,0.7698,0.9111
2,10,524,8.2591,8.1534,0.7296,0.8889
3,18,118,7.4897,7.7794,0.6826,0.7778
4,3,1056,8.0404,5.8505,0.6927,0.7556
5,9,397,7.1021,6.3411,0.5777,0.6444
6,17,959,7.1948,4.9511,0.6139,0.6222
7,16,749,6.5191,4.1404,0.5552,0.5111
8,19,153,6.3937,5.1338,0.5064,0.4667
9,6,727,6.2930,4.0537,0.5238,0.4000



Top 5 suggested third activities to make the task harder:


,activity_id,n_windows,euclid_mean,bhatt_mean,mmd_mean,DifficultyScore
0,1,993,8.6247,8.5943,0.7664,0.9778
1,2,1118,8.4851,7.7755,0.7698,0.9111
2,10,524,8.2591,8.1534,0.7296,0.8889
3,18,118,7.4897,7.7794,0.6826,0.7778
4,3,1056,8.0404,5.8505,0.6927,0.7556


In [4]:
# Rope-focused ranking (drop-in cell)
# Assumes you already ran the previous cell that created `df` with:
# ['activity_id','n_windows',
#  'euclid_to_walk','euclid_to_rope',
#  'bhattacharyya_to_walk','bhattacharyya_to_rope',
#  'mmd_to_walk','mmd_to_rope']

import numpy as np
import pandas as pd

# --- sanity check -------------------------------------------------------------
required_cols = [
    "activity_id","n_windows",
    "euclid_to_walk","euclid_to_rope",
    "bhattacharyya_to_walk","bhattacharyya_to_rope",
    "mmd_to_walk","mmd_to_rope",
]
_missing = [c for c in required_cols if c not in df.columns]
assert not _missing, f"df is missing columns: {_missing}"

# Exclude the base classes 4 (walking) and 24 (rope)
df_rf = df[~df["activity_id"].isin([4, 24])].copy()

# --- normalize each distance (per-metric min-max) -----------------------------
def _norm(col):
    rng = df_rf[col].max() - df_rf[col].min()
    return (df_rf[col] - df_rf[col].min()) / (rng + 1e-12)

# smaller distance to rope => better (we'll use 1 - rope_norm)
df_rf["euclid_rope_norm"] = _norm("euclid_to_rope")
df_rf["bhatt_rope_norm"]  = _norm("bhattacharyya_to_rope")
df_rf["mmd_rope_norm"]    = _norm("mmd_to_rope")

# larger distance to walk => better (we'll use walk_norm directly)
df_rf["euclid_walk_norm"] = _norm("euclid_to_walk")
df_rf["bhatt_walk_norm"]  = _norm("bhattacharyya_to_walk")
df_rf["mmd_walk_norm"]    = _norm("mmd_to_walk")

# --- rope-focused score -------------------------------------------------------
# Weights: emphasize closeness to rope (euclid+bhatt), then separation from walk,
# with a small MMD contribution for robustness.
w = {
    "euclid_rope": 0.30,
    "bhatt_rope":  0.30,
    "euclid_walk": 0.15,
    "bhatt_walk":  0.15,
    "mmd_rope":    0.07,
    "mmd_walk":    0.03,
}
df_rf["RopeFocusedScore"] = (
    (1 - df_rf["euclid_rope_norm"]) * w["euclid_rope"] +
    (1 - df_rf["bhatt_rope_norm"])  * w["bhatt_rope"]  +
     df_rf["euclid_walk_norm"]      * w["euclid_walk"] +
     df_rf["bhatt_walk_norm"]       * w["bhatt_walk"]  +
    (1 - df_rf["mmd_rope_norm"])    * w["mmd_rope"]    +
     df_rf["mmd_walk_norm"]         * w["mmd_walk"]
)

# --- optional: filter out tiny classes ----------------------------------------
MIN_WINDOWS = 150   # tweak if you want to allow very small classes
df_rf = df_rf[df_rf["n_windows"] >= MIN_WINDOWS].copy()

# --- attach human-readable activity names -------------------------------------
activity_names = {
    1: "lying",  2: "sitting",   3: "standing",  4: "walking",      5: "running",
    6: "cycling",7: "Nordic walking", 9: "watching TV", 10: "computer work",
    11: "car driving", 12: "ascending stairs", 13: "descending stairs",
    16: "vacuum cleaning", 17: "ironing", 18: "folding laundry",
    19: "house cleaning", 20: "playing soccer", 24: "rope jumping", 0: "other"
}
df_rf["activity_name"] = df_rf["activity_id"].map(activity_names).fillna("unknown")

# --- present top candidates ----------------------------------------------------
cols_show = [
    "activity_id","activity_name","n_windows",
    "euclid_to_rope","bhattacharyya_to_rope","mmd_to_rope",
    "euclid_to_walk","bhattacharyya_to_walk","mmd_to_walk",
    "RopeFocusedScore"
]
df_rope_ranked = df_rf.sort_values("RopeFocusedScore", ascending=False).reset_index(drop=True)

print("Top 8 rope-focused candidates (higher score = closer to rope, farther from walk):")
display(df_rope_ranked[cols_show].round(4).head(8))

# --- quick pointer for "moderate difficulty" pick -----------------------------
# Heuristic: skip the very top if it's extremely close to rope; pick the best in rank 2–4.
moderate_pick = df_rope_ranked.iloc[1:4].head(1) if len(df_rope_ranked) >= 2 else df_rope_ranked.head(1)
print("\nSuggested 'moderate difficulty' pick (rank ~2–4):")
display(moderate_pick[cols_show].round(4))


Top 8 rope-focused candidates (higher score = closer to rope, farther from walk):


,activity_id,activity_name,n_windows,euclid_to_rope,bhattacharyya_to_rope,mmd_to_rope,euclid_to_walk,bhattacharyya_to_walk,mmd_to_walk,RopeFocusedScore
0,5,running,373,3.0837,1.5053,0.1459,8.4204,4.7324,0.6705,0.9026
1,13,descending stairs,356,6.7861,2.3249,0.4745,2.0538,2.6676,0.1737,0.5094
2,12,ascending stairs,421,7.4038,2.7477,0.4987,1.6935,2.2170,0.1116,0.4473
3,6,cycling,727,9.3415,3.7065,0.6855,3.2445,4.4010,0.3620,0.4109
4,0,other,5327,9.7082,2.8389,0.5685,3.1165,1.9973,0.2668,0.3929
5,19,house cleaning,153,9.4050,4.9587,0.6513,3.3825,5.3089,0.3614,0.3764
6,7,Nordic walking,783,7.2607,4.2432,0.5880,1.2883,1.7405,0.0780,0.3555
7,16,vacuum cleaning,749,9.6497,4.5110,0.7193,3.3885,3.7698,0.3910,0.3523



Suggested 'moderate difficulty' pick (rank ~2–4):


,activity_id,activity_name,n_windows,euclid_to_rope,bhattacharyya_to_rope,mmd_to_rope,euclid_to_walk,bhattacharyya_to_walk,mmd_to_walk,RopeFocusedScore
1,13,descending stairs,356,6.7861,2.3249,0.4745,2.0538,2.6676,0.1737,0.5094


In [ ]:
# Standalone Jupyter cell to plot grouped bar charts comparing baselines across runs
# (works for binary vs multiclass splits as long as your "groups" dict provides
#  separate buckets with their own final_rows + label).
#
# Usage example (at bottom):
#   groups = {
#       "binary":    {"label": "PAMAP2 (Binary: 4 vs 24)", "final_rows": rows_bin},
#       "multiclass":{"label": "PAMAP2 (Multiclass: 4 vs 24 vs 13)", "final_rows": rows_mc},
#   }
#   plot_final_bars(groups, include_baselines=["alpha", "beta", "alpha_raworig", "alphaplusreal"])
#
# Notes:
# - No seaborn; matplotlib only.
# - One plot per group; uses matplotlib default color cycle (no explicit colors).
# - Metrics are discovered via TEST_METRICS keys; add columns like "{prefix}_{metric_key}".
# - Prefixes are detected from the left part before the first "_" in a column name.
# - "include_baselines" filters prefixes by raw name or pretty label (case/spacing-insensitive).

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import cycle

# ---------------------------- Configurable bits ---------------------------- #

# Define which metric columns to aggregate and their pretty labels.
# Each metric is expected to exist in the DataFrame as: f"{prefix}_{metric_key}"
# Example columns: "alpha_f1_minority", "beta_f1_minority", etc.
TEST_METRICS = [
    ("f1_minority",   "F1 (Rope)"),
    ("f1_majority",   "F1 (Non-Rope)"),
    ("f1_macro",      "F1 Macro"),
    ("f1_weighted",   "F1 Weighted"),
    ("accuracy",      "Accuracy"),
]

# Pretty names for common prefixes (optional; you can add more)
KNOWN_PREFIX_LABELS = {
    "alpha":          "Alpha",
    "beta":           "Beta",
    "alphact":        "Alpha+CTGAN",
    "alpha_raworig":  "Alpha_RawOrig",
    "alphaplusreal":  "Alpha+Real",
    "jitter":         "Jitter",
}

# ---------------------------- Helper functions ---------------------------- #

def _norm(s: str) -> str:
    """Normalize a string for loose matching: lowercase and strip non-alphanum."""
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

def _detect_prefixes(df: pd.DataFrame) -> list[str]:
    """
    Detect baseline prefixes by scanning columns of the form '{prefix}_{metric_key}'.
    Returns a sorted list of unique prefixes found in the DataFrame.
    """
    metric_keys = {m[0] for m in TEST_METRICS}
    prefixes = set()
    for col in df.columns:
        if "_" not in col:
            continue
        pref, key = col.split("_", 1)
        if key in metric_keys:
            prefixes.add(pref)
    return sorted(prefixes)

def _means_for_prefix(df: pd.DataFrame, prefix: str) -> np.ndarray:
    """
    For a given prefix, compute the mean across rows for each metric in TEST_METRICS.
    Returns a float array with NaN where a metric column is missing.
    """
    vals = []
    for metric_key, _pretty in TEST_METRICS:
        col = f"{prefix}_{metric_key}"
        if col in df.columns:
            vals.append(pd.to_numeric(df[col], errors="coerce").mean())
        else:
            vals.append(np.nan)
    return np.asarray(vals, dtype=float)

def plot_final_bars(groups, include_baselines=None):
    """
    Make one grouped bar chart per entry in `groups`.

    `groups` format:
      {
        "any_key": {
            "label": "Your bucket label shown on the plot",
            "final_rows": [  # list of dicts; each dict = one run/seed summary
                {"alpha_f1_minority": 0.71, "alpha_f1_macro": 0.78, ...,
                 "beta_f1_minority": 0.75, ...},
                ...
            ]
        },
        ...
      }

    include_baselines: optional list[str] of baseline names to show.
      - Matching is case-insensitive and ignores spaces/underscores/hyphens.
      - It matches against BOTH the raw prefix (e.g., 'alpha', 'alphact', 'alpha_raworig')
        and the pretty label from KNOWN_PREFIX_LABELS (e.g., 'Alpha', 'Alpha+CTGAN', 'Alpha_RawOrig').
      Example: include_baselines=['alpha', 'beta', 'alphact']
    """
    # --- spacing controls ---
    GROUP_SEP       = 1.35
    GROUP_WIDTH     = 0.96
    BAR_INNER_PAD   = 0.06
    HEADROOM        = 1.68  # extra whitespace at the top for value labels
    VALUE_ROT_DEG   = 55
    VALUE_XSHIFT_PT = 6
    VALUE_YSHIFT_PT = 6

    # Build allow-set if provided
    allow_set = None
    if include_baselines is not None:
        allow_set = { _norm(x) for x in include_baselines if str(x).strip() }

    for exp_key, bucket in groups.items():
        rows = bucket.get("final_rows", [])
        bucket_label = bucket.get("label", str(exp_key))
        if not rows:
            print(f"\nNo final_test_metrics for {bucket_label} — skipping.")
            continue

        df = pd.DataFrame(rows)
        prefixes = _detect_prefixes(df)
        if not prefixes:
            print(f"\nNo recognizable final metrics in {bucket_label} — skipping.")
            continue

        groups_bars = []
        for pref in prefixes:
            vals = _means_for_prefix(df, pref)
            if np.all(np.isnan(vals)):
                continue

            # pretty label
            label = KNOWN_PREFIX_LABELS.get(pref, pref.title())

            # filter by include_baselines (if provided)
            if allow_set is not None:
                if _norm(pref) not in allow_set and _norm(label) not in allow_set:
                    continue

            groups_bars.append((label, vals))

        if not groups_bars:
            wanted = f" {sorted(list(allow_set))}" if allow_set else ""
            print(f"\nAll detected groups empty or filtered out for {bucket_label}.{wanted}")
            continue

        # --- plotting ---
        M = len(TEST_METRICS)
        x_centers = np.arange(M) * GROUP_SEP

        n_groups = len(groups_bars)
        step = GROUP_WIDTH / max(1, n_groups)
        bar_width = max(0.04, step * (1.0 - BAR_INNER_PAD))
        offset0 = -GROUP_WIDTH / 2.0 + step / 2.0

        fig, ax = plt.subplots(figsize=(13, 5))
        bars_and_vals = []
        for gi, (glabel, gvals) in enumerate(groups_bars):
            xpos = x_centers + offset0 + gi * step
            # No explicit color (matplotlib default cycle)
            bars = ax.bar(xpos, gvals, bar_width, label=glabel)
            bars_and_vals.append((bars, gvals, glabel))

        ax.set_xticks(x_centers, [m[1] for m in TEST_METRICS])
        ax.set_ylabel("Score")
        max_val = float(np.nanmax([np.nanmax(v) for _, v in groups_bars]))
        if np.isfinite(max_val):
            ax.set_ylim(top=min(1.0, HEADROOM * max_val + 0.01))
        ax.margins(x=0.03)
        ax.grid(axis="y", alpha=0.2)
        n = len(df)
        title_core = " vs ".join(lbl for (lbl, _) in groups_bars)
        ax.set_title(f"{title_core}\nAveraged across seeds · {bucket_label} (n={n})")
        ax.legend(loc="upper left", bbox_to_anchor=(0.01, 1.02), ncol=3, frameon=False)

        def autolabel(bars, values):
            y0, y1 = ax.get_ylim()
            lift = max(0.01, 0.008 * (y1 - y0))
            for bar, val in zip(bars, values):
                if np.isnan(val):
                    continue
                x = bar.get_x() + bar.get_width() / 2
                y = bar.get_height() + lift
                ax.annotate(
                    f"{val:.3f}", (x, y),
                    xytext=(VALUE_XSHIFT_PT, VALUE_YSHIFT_PT), textcoords="offset points",
                    ha="center", va="bottom",
                    rotation=VALUE_ROT_DEG, rotation_mode="anchor",
                    bbox=dict(boxstyle="round,pad=0.15", fc="white", alpha=0.65, lw=0),
                    clip_on=False, zorder=5
                )

        for bars, vals, _ in bars_and_vals:
            autolabel(bars, vals)

        plt.tight_layout()
        plt.show()

        # --- console diffs vs anchor (first "Alpha*" if present after filtering) ---
        anchor_idx = next((i for i, (lbl, _) in enumerate(groups_bars)
                           if lbl.lower().startswith("alpha")), 0)
        anchor_label, anchor_vals = groups_bars[anchor_idx]
        print(f"\nComparisons vs {anchor_label} · averaged across seeds · {bucket_label}:")
        for lbl, vals in groups_bars:
            if lbl == anchor_label:
                continue
            print(f"  {lbl}:")
            for j, (_, metric_name) in enumerate(TEST_METRICS):
                a = anchor_vals[j]; b = vals[j]
                if np.isnan(a) or np.isnan(b) or a == 0:
                    print(f"    {metric_name}: N/A")
                else:
                    diff = b - a
                    pct = 100.0 * diff / abs(a)
                    print(f"    {metric_name}: {diff:+.3f} ({pct:+.1f}%)")

# ---------------------------- Example (commented) ---------------------------- #
# Suppose you collected per-seed final metrics into two lists of dicts:
rows_bin = [
    {"alpha_f1_minority": 0.74, "alpha_f1_majority": 0.91, "alpha_f1_macro": 0.83, "alpha_f1_weighted": 0.88, "alpha_accuracy": 0.89,
     "beta_f1_minority":  0.77, "beta_f1_majority":  0.92, "beta_f1_macro":  0.85, "beta_f1_weighted":  0.90, "beta_accuracy":  0.91},
    # ... more seeds
]
rows_mc = [
    {"alpha_f1_minority": 0.62, "alpha_f1_majority": 0.88, "alpha_f1_macro": 0.75, "alpha_f1_weighted": 0.84, "alpha_accuracy": 0.85,
     "beta_f1_minority":  0.66, "beta_f1_majority":  0.89, "beta_f1_macro":  0.78, "beta_f1_weighted":  0.86, "beta_accuracy":  0.87},
    # ... more seeds
]
groups = {
    "binary":     {"label": "PAMAP2 (Binary: 4 vs 24)",             "final_rows": rows_bin},
    "multiclass": {"label": "PAMAP2 (Multiclass: 4 vs 24 vs 13)",   "final_rows": rows_mc},
}
# plot_final_bars(groups, include_baselines=["alpha", "beta"])


In [6]:
plot_final_bars(groups, include_baselines=["alpha", "beta", "alpha_raworig", "alphaplusreal"])

NameError: name 'groups' is not defined